In [20]:
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd


training_df = pd.read_csv('landsat_features_training_ross.csv')

submission_df = pd.read_csv('landsat_features_validation_ross.csv')

for col in training_df.columns:
    training_df = training_df.rename(columns={col: col.lower()})

training_df['sample date'] = pd.to_datetime(training_df['sample date'],format='%d-%m-%Y')

training_df.head(5)



for col in submission_df.columns:
    submission_df = submission_df.rename(columns={col: col.lower()})

submission_df['sample date'] = pd.to_datetime(submission_df['sample date'],format='%d-%m-%Y')


In [21]:
training_df.columns

Index(['latitude', 'longitude', 'sample date', 'emis', 'atran', 'swir16',
       'atmos_opacity', 'green', 'drad', 'red', 'swir22', 'emsd', 'cdist',
       'nir08', 'blue', 'trad', 'lwir', 'urad', 'nvdi', 'savi', 'bsi', 'ndbi',
       'tcwi'],
      dtype='str')

In [22]:
B2	= 'blue'
B3	= 'green'
B4	= 'red'
B5	= 'nir08'
B6	= 'swir16'
B7	= 'swir22'


In [23]:
original_set = [training_df, submission_df]

#original_set[i][B5]

for i in range(len(original_set)):

    B2	= original_set[i]['blue']
    B3	= original_set[i]['green']
    B4	= original_set[i]['red']
    B5	= original_set[i]['nir08']
    B6	= original_set[i]['swir16']
    B7	= original_set[i]['swir22']
    
    original_set[i]['evi'] = 2.5 * ((B5 - B4) / (B5 + 6*B4 - 7.5*B2 + 1))
    original_set[i]['osavi']	= 1.16 * (B5 - B4) / (B5 + B4 + 0.16)
    original_set[i]['gndvi']	= (B5 - B3) / (B5 + B3)
    original_set[i]['gcvi']	= (B5 / B3) - 1
    original_set[i]['msi']	= B6 / B5
    original_set[i]['bui']	= original_set[i]['ndbi'] - original_set[i]['nvdi']
    original_set[i]['tc brightness'] =	0.3029*B2 + 0.2786*B3 + 0.4733*B4 + 0.5599*B5 + 0.5080*B6 + 0.1872*B7
    original_set[i]['tc greenness'] =	-0.2941*B2 -0.2430*B3 -0.5424*B4 +0.7276*B5 +0.0713*B6 -0.1608*B7

    original_set[i]['nbr']=	(B5 - B7) / (B5 + B7)
    original_set[i]['ndsi']= 	(B3 - B6) / (B3 + B6)
    original_set[i]['green/red ratio'] =	B3 / B4
    original_set[i]['ndgi'] =	(B3 - B4) / (B3 + B4)
    original_set[i]['ui (urban index)'] =	((B6/B5) - (B4/B3))
    original_set[i]['nbr2'] =	(B6 - B7) / (B6 + B7)
    original_set[i]['red/nir ratio'] =	B4 / B5
    original_set[i]['green/nir ratio'] =	B3 / B5


In [24]:
training_df.head(5)

,latitude,longitude,sample date,emis,atran,swir16,atmos_opacity,green,drad,red,...,tc brightness,tc greenness,nbr,ndsi,green/red ratio,ndgi,ui (urban index),nbr2,red/nir ratio,green/nir ratio
0,-28.760833,17.730278,02-01-2011,9880.0,7669.0,7687.5,8.0,11426.0,890.0,12802.0,...,23738.96050,-5070.38975,0.188213,0.195595,0.892517,-0.056794,-0.433430,0.002772,1.144057,1.021090
1,-26.861111,28.884722,03-01-2011,9570.5,7957.5,13746.5,46.0,9550.0,747.0,9241.5,...,28516.80480,2238.88815,0.250934,-0.180134,1.033382,0.016417,-0.189233,0.130446,0.523346,0.540816
2,-26.450000,28.085833,03-01-2011,9670.5,9051.0,17974.0,9.0,10720.0,329.0,12540.0,...,32105.37945,-2136.51985,0.034307,-0.252805,0.854864,-0.078246,0.011946,0.117265,0.824458,0.704799
3,-27.671111,27.236944,03-01-2011,9756.5,6476.0,13522.0,48.0,10943.0,1198.0,11237.5,...,28642.63860,-1642.92995,0.132522,-0.105416,0.973793,-0.013277,-0.118603,0.085015,0.754853,0.735071
4,-27.356667,27.286389,03-01-2011,9829.5,6521.0,12665.5,106.5,9502.5,1187.0,9290.0,...,27346.55645,1684.90505,0.271443,-0.142683,1.022874,0.011308,-0.225015,0.135486,0.552040,0.564667


In [25]:
training_df.to_csv('landsat_features_training_mvdb.csv', index=False)
submission_df.to_csv('landsat_features_validation_mvdb.csv', index=False)

In [ ]:
session.sql("""
    PUT file://landsat_features_training_mvdb.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/landsat'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

In [ ]:
session.sql("""
    PUT file://landsat_features_validation_mvdb.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/landsat'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")